# Ultimate Technologies Data Science Challenge

In [1]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

OUTDIR = Path("./output")
OUTDIR.mkdir(exist_ok=True)

OUTDIR


WindowsPath('output')

## Part 1 — Exploratory data analysis

Aggregate login counts into 15-minute buckets, then summarize the main temporal patterns.


In [4]:
def part1_login_analysis(logins_path: str) -> pd.DataFrame:
    with open(logins_path) as f:
        logins = json.load(f)
    logins_df = pd.DataFrame(logins)
    logins_df['login_time'] = pd.to_datetime(logins_df['login_time'])

    agg = (
        logins_df.set_index('login_time')
        .sort_index()
        .resample('15min')
        .size()
        .rename('count')
        .to_frame()
    )
    agg['hour'] = agg.index.hour + agg.index.minute / 60
    agg['dayofweek'] = agg.index.day_name()
    agg['is_weekend'] = agg.index.dayofweek >= 5

    # Figure 1: time series
    plt.figure(figsize=(11, 4))
    plt.plot(agg.index, agg['count'], linewidth=0.8)
    plt.title('Login counts by 15-minute interval')
    plt.xlabel('Time')
    plt.ylabel('Logins')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'part1_timeseries.png', dpi=180)
    plt.close()

    # Figure 2: heatmap
    pivot = (
        agg.pivot_table(index='dayofweek', columns='hour', values='count', aggfunc='mean')
        .reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
    )
    plt.figure(figsize=(12, 4.5))
    plt.imshow(pivot, aspect='auto', interpolation='nearest')
    plt.colorbar(label='Average logins per 15 min')
    plt.yticks(range(len(pivot.index)), pivot.index)
    xticks = np.arange(0, len(pivot.columns), 4)
    plt.xticks(
        xticks,
        [f'{int(pivot.columns[i])}:00' if abs(pivot.columns[i] - int(pivot.columns[i])) < 1e-9 else '' for i in xticks],
        rotation=45,
    )
    plt.title('Average login intensity by day of week and hour')
    plt.xlabel('Hour of day')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'part1_heatmap.png', dpi=180)
    plt.close()

    # Figure 3: weekday vs weekend
    weekday_hourly = agg[~agg['is_weekend']].groupby('hour')['count'].mean()
    weekend_hourly = agg[agg['is_weekend']].groupby('hour')['count'].mean()
    plt.figure(figsize=(10, 4))
    plt.plot(weekday_hourly.index, weekday_hourly.values, label='Weekday')
    plt.plot(weekend_hourly.index, weekend_hourly.values, label='Weekend')
    plt.title('Average login pattern: weekday vs weekend')
    plt.xlabel('Hour of day')
    plt.ylabel('Average logins per 15 min')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTDIR / 'part1_weekday_weekend.png', dpi=180)
    plt.close()

    return agg




In [6]:
agg = part1_login_analysis('logins.json')

daily_totals = agg['count'].groupby(agg.index.date).sum()
weekday_hourly = agg[~agg['is_weekend']].groupby('hour')['count'].mean()
weekend_hourly = agg[agg['is_weekend']].groupby('hour')['count'].mean()

part1_summary = {
    'records': 93142,
    'intervals': int(len(agg)),
    'mean_15m': round(float(agg['count'].mean()), 3),
    'median_15m': round(float(agg['count'].median()), 3),
    'peak_interval': int(agg['count'].max()),
    'peak_timestamp': str(agg['count'].idxmax()),
    'weekday_peak_hour': float(weekday_hourly.idxmax()),
    'weekend_peak_hour': float(weekend_hourly.idxmax()),
    'first_day_total': int(daily_totals.iloc[0]),
    'last_day_total': int(daily_totals.iloc[-1]),
}

part1_summary


{'records': 93142,
 'intervals': 9788,
 'mean_15m': 9.516,
 'median_15m': 7.0,
 'peak_interval': 73,
 'peak_timestamp': '1970-03-01 04:30:00',
 'weekday_peak_hour': 11.5,
 'weekend_peak_hour': 4.75,
 'first_day_total': 112,
 'last_day_total': 395}

## Part 3 — Predictive modeling

Define retained users, clean the dataset, compare two baseline models, and summarize key performance metrics.

**Retention assumption:** users are considered retained if their last trip date is on or after **2014-06-01**, corresponding to an assumed data pull date of **2014-07-01**.


In [7]:
def part3_retention_analysis(data_path: str) -> dict:
    with open(data_path) as f:
        rows = json.load(f)
    df = pd.DataFrame(rows)
    for col in ['signup_date', 'last_trip_date']:
        df[col] = pd.to_datetime(df[col])

    # Standard assumption for this challenge: pulled on 2014-07-01.
    cutoff = pd.Timestamp('2014-06-01')
    df['retained'] = (df['last_trip_date'] >= cutoff).astype(int)

    # EDA plots
    city = df.groupby('city')['retained'].mean().sort_values()
    plt.figure(figsize=(7, 4))
    plt.barh(city.index, city.values)
    plt.title('Retention rate by city')
    plt.xlabel('Retention rate')
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(OUTDIR / 'part3_retention_by_city.png', dpi=180)
    plt.close()

    phone = df.groupby('phone')['retained'].mean().sort_values()
    plt.figure(figsize=(6, 3.6))
    plt.barh([str(x) for x in phone.index], phone.values)
    plt.title('Retention rate by phone type')
    plt.xlabel('Retention rate')
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(OUTDIR / 'part3_retention_by_phone.png', dpi=180)
    plt.close()

    df['signup_day'] = df['signup_date'].dt.day
    df['signup_weekday'] = df['signup_date'].dt.day_name()
    features = [
        'city',
        'phone',
        'avg_dist',
        'avg_rating_by_driver',
        'avg_rating_of_driver',
        'avg_surge',
        'surge_pct',
        'trips_in_first_30_days',
        'ultimate_black_user',
        'weekday_pct',
        'signup_day',
        'signup_weekday',
    ]
    X = df[features]
    y = df['retained']

    cat = ['city', 'phone', 'ultimate_black_user', 'signup_weekday']
    num = [c for c in features if c not in cat]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    prep = ColumnTransformer(
        [
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num),
            (
                'cat',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('ohe', OneHotEncoder(handle_unknown='ignore')),
                ]),
                cat,
            ),
        ]
    )

    estimators = {
        'Logistic regression': LogisticRegression(max_iter=2000),
        'Random forest': RandomForestClassifier(
            n_estimators=120, min_samples_leaf=5, random_state=42, n_jobs=-1
        ),
    }

    model_metrics = {}
    for name, estimator in estimators.items():
        model = Pipeline([('prep', prep), ('model', estimator)])
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        model_metrics[name] = {
            'roc_auc': roc_auc_score(y_test, prob),
            'avg_precision': average_precision_score(y_test, prob),
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred),
            'recall': recall_score(y_test, pred),
            'f1': f1_score(y_test, pred),
            'brier': brier_score_loss(y_test, prob),
        }

    metric_df = pd.DataFrame(model_metrics).T[['roc_auc', 'avg_precision', 'accuracy', 'f1']]
    metric_df.plot(kind='bar', figsize=(8, 4), rot=0)
    plt.ylim(0, 1)
    plt.title('Model comparison on holdout set')
    plt.ylabel('Score')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'part3_model_comparison.png', dpi=180)
    plt.close()

    return {
        'retention_rate': float(df['retained'].mean()),
        'city_retention': df.groupby('city')['retained'].mean().round(4).to_dict(),
        'phone_retention': df.groupby('phone')['retained'].mean().round(4).to_dict(),
        'missing_rate': df.isna().mean().round(4).sort_values(ascending=False).to_dict(),
        'model_metrics': {k: {m: round(float(v), 4) for m, v in d.items()} for k, d in model_metrics.items()},
    }

In [8]:
retention_summary = part3_retention_analysis('ultimate_data_challenge.json')
retention_summary


{'retention_rate': 0.37608,
 'city_retention': {'Astapor': 0.2557,
  "King's Landing": 0.6281,
  'Winterfell': 0.3519},
 'phone_retention': {'Android': 0.2094, 'iPhone': 0.4489},
 'missing_rate': {'avg_rating_of_driver': 0.1624,
  'phone': 0.0079,
  'avg_rating_by_driver': 0.004,
  'signup_date': 0.0,
  'trips_in_first_30_days': 0.0,
  'avg_surge': 0.0,
  'city': 0.0,
  'last_trip_date': 0.0,
  'surge_pct': 0.0,
  'weekday_pct': 0.0,
  'ultimate_black_user': 0.0,
  'avg_dist': 0.0,
  'retained': 0.0,
  'signup_day': 0.0,
  'signup_weekday': 0.0},
 'model_metrics': {'Logistic regression': {'roc_auc': 0.7535,
   'avg_precision': 0.6588,
   'accuracy': 0.7165,
   'precision': 0.6676,
   'recall': 0.4903,
   'f1': 0.5654,
   'brier': 0.1895},
  'Random forest': {'roc_auc': 0.8503,
   'avg_precision': 0.7835,
   'accuracy': 0.7824,
   'precision': 0.7454,
   'recall': 0.64,
   'f1': 0.6887,
   'brier': 0.1499}}}

## Save final summary

This writes the key summary statistics used in the report to `summary.json`.


In [9]:
summary = {
    'part1': part1_summary,
    'part3': retention_summary,
}

with open(OUTDIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

summary


{'part1': {'records': 93142,
  'intervals': 9788,
  'mean_15m': 9.516,
  'median_15m': 7.0,
  'peak_interval': 73,
  'peak_timestamp': '1970-03-01 04:30:00',
  'weekday_peak_hour': 11.5,
  'weekend_peak_hour': 4.75,
  'first_day_total': 112,
  'last_day_total': 395},
 'part3': {'retention_rate': 0.37608,
  'city_retention': {'Astapor': 0.2557,
   "King's Landing": 0.6281,
   'Winterfell': 0.3519},
  'phone_retention': {'Android': 0.2094, 'iPhone': 0.4489},
  'missing_rate': {'avg_rating_of_driver': 0.1624,
   'phone': 0.0079,
   'avg_rating_by_driver': 0.004,
   'signup_date': 0.0,
   'trips_in_first_30_days': 0.0,
   'avg_surge': 0.0,
   'city': 0.0,
   'last_trip_date': 0.0,
   'surge_pct': 0.0,
   'weekday_pct': 0.0,
   'ultimate_black_user': 0.0,
   'avg_dist': 0.0,
   'retained': 0.0,
   'signup_day': 0.0,
   'signup_weekday': 0.0},
  'model_metrics': {'Logistic regression': {'roc_auc': 0.7535,
    'avg_precision': 0.6588,
    'accuracy': 0.7165,
    'precision': 0.6676,
    'reca